In [1]:
import re
import os
import json
import pandas as pd
import numpy as np
import sys
import requests
from pathlib import Path
import mygene
from typing import Dict, List, Tuple, Set

In [2]:
#set current working directory to the library path
library_path = Path('/villa/rhh25/Cellhit/') #replace with yours
os.chdir(library_path)

In [3]:
ccle_expression = pd.read_csv('data/ccle/OmicsExpressionExpectedCountHumanAllGenes.csv', index_col=0)
df_drugcombs = pd.read_csv('data/metadata/drugcombs_scored.csv')
metadata = pd.read_csv("data/ccle/Model.csv")

In [10]:
df_drugcombs

,ID,Drug1,Drug2,Cell line,ZIP,Bliss,Loewe,HSA,CellLineClean,ModelID
0,1,5-FU,ABT-888,A2058,1.720,6.260,-2.750,5.540,A2058,ACH-000788
1,2,5-FU,ABT-888,A2058,5.880,12.330,3.330,11.610,A2058,ACH-000788
2,3,5-FU,ABT-888,A2058,3.590,11.660,2.650,10.940,A2058,ACH-000788
3,4,5-FU,ABT-888,A2058,-0.850,5.150,-3.860,4.430,A2058,ACH-000788
4,5,5-FU,AZD1775,A2058,12.290,15.770,10.400,18.660,A2058,ACH-000788
...,...,...,...,...,...,...,...,...,...,...
498860,498861,Mitomycin C,Valproic acid sodium salt,DIPG25,-3.076,-11.124,-6.078,-6.367,DIPG25,None
498861,498862,Cyanein,Valproic acid sodium salt,DIPG25,1.320,-1.091,-16.157,1.139,DIPG25,None
498862,498863,Erlotinib,Valproic acid sodium salt,DIPG25,-15.768,-15.776,-25.465,-6.212,DIPG25,None
498863,498864,Bafilomycin A1,Valproic acid sodium salt,DIPG25,2.208,10.567,-9.455,1.525,DIPG25,None


In [4]:
ccle_expression

,SequencingID,ModelID,IsDefaultEntryForModel,ModelConditionID,IsDefaultEntryForMC,TSPAN6 (7105),TNMD (64102),DPM1 (8813),SCYL3 (57147),FIRRM (55732),...,ENSG00000288716.1,ENSG00000288717.1,ENSG00000288718.1,ENSG00000288719.1,ENSG00000288720.1,ENSG00000288721.1,F8A1 (8263),ENSG00000288723.1,ENSG00000288724.1,ENSG00000288725.1
0,CDS-010xbm,ACH-001113,Yes,MC-001113-k2lR,Yes,2099,0,4466,919,1149,...,0,1,0,3,19,55,352,0,0,0
1,CDS-02TzJp,ACH-001289,Yes,MC-001289-BpdI,Yes,2226,12,3244,525,636,...,0,2,0,1,9,18,374,0,0,5
2,CDS-0693hw,ACH-001339,Yes,MC-001339-5nRN,Yes,1355,0,6900,823,1934,...,0,0,3,0,14,15,207,1,0,0
3,CDS-07Plat,ACH-001619,No,MC-001619-IR6I,No,3410,0,2109,414,432,...,0,0,0,0,5,15,991,0,0,0
4,CDS-08FOcu,ACH-001979,Yes,MC-001979-E3qW,Yes,2578,0,1708,667,193,...,0,1,0,0,7,11,638,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1749,CDS-zsS3zE,ACH-002669,Yes,MC-002669-Ivra,Yes,1329,0,5297,390,818,...,0,2,37,0,25,22,3670,10,0,0
1750,CDS-zsp0NA,ACH-001858,Yes,MC-001858-c43i,Yes,4192,0,6635,796,820,...,0,0,11,2,5,72,2406,15,0,0
1751,CDS-zvrgDc,ACH-001997,Yes,MC-001997-n8wz,Yes,36457,0,57710,6898,8278,...,0,0,18,6,229,260,9956,11,0,9
1752,CDS-zvw4zv,ACH-003297,Yes,MC-003297-yuio,Yes,3358,0,12597,385,927,...,0,0,0,0,23,27,227,0,0,0


In [9]:
metadata

,ModelID,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
0,ACH-000001,PT-gj46wT,NIH:OVCAR-3,NIHOVCAR3,HGSOC,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,HGSOC,NaN,...,NaN,NIHOVCAR3_OVARY,NaN,False,Approved for public sharing - CCLE,NaN,2201.0,SIDM00105,905933.0,NaN
1,ACH-000002,PT-5qa3uk,HL-60,HL60,AMLMRC,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes,AMLMRC,"TP53(del), CDKN2A and NRAS mutations [PubMed=2...",...,NaN,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"NRAS, BCOR and CDKN2A",55.0,SIDM00829,905938.0,NaN
2,ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
3,ACH-000004,PT-q4K2cp,HEL,HEL,AMLNOS,Myeloid,Acute Myeloid Leukemia,"AML, NOS",AMLNOS,JAK2 and TP53 mutations,...,NaN,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,JAK2 and TP53,783.0,SIDM00594,907053.0,NaN
4,ACH-000005,PT-q4K2cp,HEL 92.1.7,HEL9217,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,JAK2 and TP53 mutations,...,NaN,HEL9217_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,NaN,NaN,SIDM00593,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2127,ACH-003473,PT-fG5tCh,CCLF_PEDS_0013_T,CCLFPEDS0013T,ERMS,Soft Tissue,Rhabdomyosarcoma,Embryonal Rhabdomyosarcoma,ERMS,NaN,...,NaN,NaN,HCM-BROD-0006-C49,True,NaN,NaN,NaN,NaN,NaN,NaN
2128,ACH-003474,PT-WxfjG3,CCLF_HNSC_0001_T,CCLFHNSC0001T,HNSC,Head and Neck,Head and Neck Squamous Cell Carcinoma,Head and Neck Squamous Cell Carcinoma,HNSC,NaN,...,NaN,NaN,HCM-BROD-1131-C06,False,NaN,NaN,NaN,NaN,NaN,NaN
2129,ACH-003475,PT-ce6oqw,CCLF_HNSC_0003_T,CCLFHNSC0003T,HNSC,Head and Neck,Head and Neck Squamous Cell Carcinoma,Head and Neck Squamous Cell Carcinoma,HNSC,NaN,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN
2130,ACH-003476,PT-ce6oqw,CCLF_HNSC_0002_T,CCLFHNSC0002T,ESCC,Esophagus/Stomach,Esophageal Squamous Cell Carcinoma,Esophageal Squamous Cell Carcinoma,ESCC,NaN,...,NaN,NaN,HCM-BROD-1130-C06,False,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
cellline_other_info = ['SequencingID', 'IsDefaultEntryForModel', 'ModelConditionID', 'IsDefaultEntryForMC']

In [6]:
def map_cellline_to_modelid(cellline_name, mapping_dict):
    """Map DrugComb cell line name to DepMap ModelID"""
    cellline_upper = cellline_name.upper()
    
    # Exact match
    if cellline_upper in mapping_dict:
        return mapping_dict[cellline_upper]
    
    # Partial match (if exact not found)
    for mapped_name, model_id in mapping_dict.items():
        if cellline_upper in mapped_name or mapped_name in cellline_upper:
            return model_id
    
    return None


def get_active_genes(df_drugcombs, metadata, ccle_expression, threshold=1000):
    """Identify active genes based on mean expression across cell lines"""
    # Create mapping: StrippedCellLineName -> ModelID
    cell_line_list = df_drugcombs['Cell line'].unique()
    print("Number of unique cell lines in DrugComb:", len(cell_line_list))

    cellline_to_modelid = dict(zip(
        metadata['StrippedCellLineName'].str.upper(),
        metadata['ModelID']
    ))


    df_drugcombs['CellLineClean'] = df_drugcombs['Cell line'].apply(lambda x: re.sub(r'[-\s\/\\]', '', x.strip().upper()))
    df_drugcombs['CellLineClean'] = df_drugcombs['CellLineClean'].apply(lambda x: 'HL60' if x in ['HL60(TB)'] else x)

    df_drugcombs['ModelID'] = df_drugcombs['CellLineClean'].apply(
        lambda x: map_cellline_to_modelid(x, cellline_to_modelid)
    )

    # Extract metadata columns
    metadata_cols = ['SequencingID', 'ModelID', 'IsDefaultEntryForModel', 
                    'ModelConditionID', 'IsDefaultEntryForMC']

    # Extract gene expression columns (everything else)
    expression_cols = [col for col in ccle_expression.columns if col not in metadata_cols]

    print(f"Gene expression columns: {len(expression_cols)}")
    ccle_expression_filter = ccle_expression[ccle_expression['ModelID'].isin(df_drugcombs['ModelID'].dropna().unique())].set_index('ModelID')[expression_cols]

    active_genes =  ccle_expression_filter.columns[ccle_expression_filter.mean(numeric_only=True) > threshold].tolist()
    print(f"Number of active genes: {len(active_genes)}")


    number_celllines = len(ccle_expression_filter.T.columns[ccle_expression_filter[active_genes].T.mean(numeric_only=True)>0])
    print("Number of cell lines with active genes: ", number_celllines)


    ## remove parentheses in and content in columns name
    ccle_expression_filter = ccle_expression_filter[active_genes]
    ccle_expression_filter.columns = ccle_expression_filter.columns.str.replace(r'\(.*\)', '', regex=True).str.strip()

    active_genes = ccle_expression_filter.columns.tolist()
    return active_genes, ccle_expression_filter



In [5]:
def map_ensembl_proteins_to_genes(ensembl_protein_ids):
    """
    Convert Ensembl protein IDs (ENSP...) to gene symbols
    
    Input format: ['9606.ENSP00000002233', '9606.ENSP00000000412', ...]
    Output: {'ENSP00000002233': 'TSPAN6', 'ENSP00000000412': 'DPM1', ...}
    """
    print(f"\nMapping {len(ensembl_protein_ids)} Ensembl proteins to genes...")
    
    ensp_ids = [p.split('.')[-1] for p in ensembl_protein_ids]
    
    mg = mygene.MyGeneInfo()
    
    protein_to_gene = {}
    
    # Process in batches
    batch_size = 300
    for i in range(0, len(ensp_ids), batch_size):
        batch = ensp_ids[i:i+batch_size]
        
        try:
            # Query by Ensembl protein ID
            results = mg.querymany(
                batch,
                scopes='ensemblprotein',  # Query by Ensembl protein ID
                species='human',
                fields='symbol',  # Get gene symbol
                as_dataframe=False,
                returnall=False
            )
            
            for result in results:
                if 'symbol' in result:
                    ensp_id = f'9606.{result["query"]}'
                    gene_symbol = result['symbol']
                    protein_to_gene[ensp_id] = gene_symbol
        
        except Exception as e:
            #print(f"Error in batch {i//batch_size}: {e}")
            continue
    
    with open("data/metadata/protein_to_gene_mapping.json", "w") as f:
        json.dump(protein_to_gene, f, indent=4)

    print(f"Successfully mapped {len(protein_to_gene)} proteins to genes")
    
    return protein_to_gene


In [7]:
active_genes, ccle_active_expression = get_active_genes(df_drugcombs, metadata, ccle_expression, threshold=1000)

Number of unique cell lines in DrugComb: 123
Gene expression columns: 60649
Number of active genes: 9601
Number of cell lines with active genes:  93


In [8]:
ccle_active_expression

,TSPAN6,DPM1,FIRRM,CFH,FUCA2,GCLC,NFYA,NIPAL3,LAS1L,SEMA3F,...,ENSG00000285437.1,ENSG00000285600.1,ENSG00000285608.1,ENSG00000285756.2,ENSG00000286037.1,ENSG00000286070.2,DERPC,DUS4L-BCAP29,PRRC2B,F8A1
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000421,1565,2709,1087,10,5577,3001,1253,1067,2233,664,...,1027,796,1541,2615,3200,2262,1030,271,10030,3613
ACH-000754,3,2494,1167,4,1354,1901,1835,827,6319,3,...,2517,3182,1364,1892,3148,3630,975,578,9919,2482
ACH-000376,1636,3913,1286,5022,10607,3065,3170,3196,7550,327,...,2236,1034,503,491,14,2894,799,2015,12995,1033
ACH-000035,2723,7363,1214,54,8534,2115,1980,3044,4376,3074,...,810,1648,2541,1330,14112,4185,1464,463,23096,1429
ACH-000322,435,7263,970,116,5354,1348,3030,2726,2973,142,...,371,577,2701,528,4,3276,1167,1191,10006,107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000900,2052,8077,878,90,901,3150,3602,566,12298,92,...,1598,2774,1492,1688,803,2736,1355,1498,14852,1523
ACH-000115,977,3712,1125,9,4207,4510,6774,5573,6109,1269,...,5428,1580,2699,2726,4220,6123,430,1683,38503,1584
ACH-000046,1373,2205,772,109,9654,2156,1495,572,2470,2181,...,904,806,1640,419,44,3098,2026,1627,10229,897


In [89]:
len(active_genes)

9601

In [108]:
ccle_active_expression

,TSPAN6,DPM1,FIRRM,CFH,FUCA2,GCLC,NFYA,NIPAL3,LAS1L,SEMA3F,...,ENSG00000285437.1,ENSG00000285600.1,ENSG00000285608.1,ENSG00000285756.2,ENSG00000286037.1,ENSG00000286070.2,DERPC,DUS4L-BCAP29,PRRC2B,F8A1
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000421,1565,2709,1087,10,5577,3001,1253,1067,2233,664,...,1027,796,1541,2615,3200,2262,1030,271,10030,3613
ACH-000754,3,2494,1167,4,1354,1901,1835,827,6319,3,...,2517,3182,1364,1892,3148,3630,975,578,9919,2482
ACH-000376,1636,3913,1286,5022,10607,3065,3170,3196,7550,327,...,2236,1034,503,491,14,2894,799,2015,12995,1033
ACH-000035,2723,7363,1214,54,8534,2115,1980,3044,4376,3074,...,810,1648,2541,1330,14112,4185,1464,463,23096,1429
ACH-000322,435,7263,970,116,5354,1348,3030,2726,2973,142,...,371,577,2701,528,4,3276,1167,1191,10006,107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000900,2052,8077,878,90,901,3150,3602,566,12298,92,...,1598,2774,1492,1688,803,2736,1355,1498,14852,1523
ACH-000115,977,3712,1125,9,4207,4510,6774,5573,6109,1269,...,5428,1580,2699,2726,4220,6123,430,1683,38503,1584
ACH-000046,1373,2205,772,109,9654,2156,1495,572,2470,2181,...,904,806,1640,419,44,3098,2026,1627,10229,897


In [11]:
ppi_file_path = 'data/ppi/9606.protein.links.full.v12.0.txt' 
print(f"Loading PPI data from {ppi_file_path}...")

# Read the PPI file
# Format: protein1 \t protein2 \t combined_score
ppi_data = pd.read_csv(ppi_file_path, sep=' ', header=None,  skiprows=1,
                        names=['protein1', 'protein2', 'neighborhood', 'neighborhood_transferred', 'fusion', 'cooccurence', 'homology', 'coexpression', 'coexpression_transferred', 'experiments', 'experiments_transferred', 'database', 'database_transferred', 'textmining', 'textmining_transferred', 'combined_score'])

all_proteins = list(set(ppi_data['protein1'].tolist()).union(set(ppi_data['protein2'].tolist())))

Loading PPI data from data/ppi/9606.protein.links.full.v12.0.txt...


In [32]:
ppi_data[['protein1', 'protein2','combined_score']]

,protein1,protein2,combined_score
0,9606.ENSP00000000233,9606.ENSP00000356607,173
1,9606.ENSP00000000233,9606.ENSP00000427567,154
2,9606.ENSP00000000233,9606.ENSP00000253413,151
3,9606.ENSP00000000233,9606.ENSP00000493357,471
4,9606.ENSP00000000233,9606.ENSP00000324127,201
...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169


In [33]:
protein_to_genes = map_ensembl_proteins_to_genes(all_proteins)


Mapping 19622 Ensembl proteins to genes...


1 input query terms found dup hits:	[('ENSP00000319979', 2)]
7 input query terms found no hit:	['ENSP00000277895', 'ENSP00000498241', 'ENSP00000405899', 'ENSP00000493046', 'ENSP00000393854', 'ENS
4 input query terms found no hit:	['ENSP00000321360', 'ENSP00000491657', 'ENSP00000479202', 'ENSP00000376568']
8 input query terms found no hit:	['ENSP00000464577', 'ENSP00000342836', 'ENSP00000305403', 'ENSP00000376103', 'ENSP00000351807', 'ENS
8 input query terms found no hit:	['ENSP00000252655', 'ENSP00000323688', 'ENSP00000479606', 'ENSP00000485615', 'ENSP00000386772', 'ENS
13 input query terms found no hit:	['ENSP00000444339', 'ENSP00000357714', 'ENSP00000439228', 'ENSP00000364858', 'ENSP00000403660', 'ENS
11 input query terms found no hit:	['ENSP00000497716', 'ENSP00000461879', 'ENSP00000335156', 'ENSP00000493701', 'ENSP00000498759', 'ENS
3 input query terms found no hit:	['ENSP00000359563', 'ENSP00000433642', 'ENSP00000473444']
10 input query terms found no hit:	['ENSP00000391381', 'ENS

Successfully mapped 18963 proteins to genes


In [34]:
## Get active PPI data

protein_gene_df = pd.DataFrame(list(protein_to_genes.items()), columns=["protein", "gene"])
active_protein_gene_df = protein_gene_df[protein_gene_df['gene'].isin(active_genes)]
active_proteins = active_protein_gene_df['protein'].unique()
active_ppi_data = ppi_data[ppi_data['protein1'].isin(active_proteins) & ppi_data['protein2'].isin(active_proteins)]

In [35]:
print(f"Number of original active genes: {len(active_genes)}")
new_active_genes = active_protein_gene_df['gene'].unique()
print(f"Number of active genes after PPI filtering: {len(new_active_genes)}")

cellline_expression_data = ccle_active_expression[new_active_genes]

cellline_expression_data.to_csv("data/graph_data/cell_line_expression.csv", index=False)
active_ppi_data.to_csv("data/graph_data/active_ppi_data.csv", index=False)

with open("data/graph_data/protein_to_gene_mapping.json", "w") as f:
        json.dump(protein_to_genes, f, indent=4)

Number of original active genes: 9601
Number of active genes after PPI filtering: 9134


In [37]:
active_ppi_data.protein1.nunique()

9141

In [121]:
df_drugcombs[['Cell line', 'ModelID']].dropna().drop_duplicates().reset_index(drop=True)

cellline_to_model_dict = dict(zip(df_drugcombs['Cell line'], df_drugcombs['ModelID']))

with open("data/graph_data/cellline_to_model_mapping.json", "w") as f:
        json.dump(cellline_to_model_dict, f, indent=4)

### Drugs

In [16]:
path = "data/drugs/interactions.tsv"
drug_df = pd.read_csv(path, sep='\t', low_memory=False)

In [17]:
all_drugs = set(df_drugcombs.Drug1.unique()).union(set(df_drugcombs.Drug2.unique()))

In [18]:
def build_drug_to_genes_mapping(dgidb_interactions):
    """
    Create a mapping: Drug → List of target genes
    
    Returns:
        drug_to_genes: {drug_name_normalized: [gene1, gene2, ...]}
        gene_to_drugs: {gene: [drug1, drug2, ...]}
    """
    print("\nBuilding drug-to-genes mapping...")
    

    drug_col = 'drug_name_normalized'  # or 'Drug_name', 'drugName'
    gene_col = 'gene_name'  # or 'Gene_name', 'geneName'
    
    # Create mappings
    drug_to_genes = {}
    
    for _, row in dgidb_interactions.iterrows():
        drug = str(row[drug_col]).strip()
        gene = str(row[gene_col]).strip()
        
        # Drug -> Genes
        if drug not in drug_to_genes:
            drug_to_genes[drug] = set()
        drug_to_genes[drug].add(gene)
        
    drug_to_genes = {k: list(v) for k, v in drug_to_genes.items()}

    
    print(f"Mapped {len(drug_to_genes)} drugs to genes")
    
    return drug_to_genes

def normalize_drug_id(drug_id):
    """
    Normalize drug ID to handle multiple formats:
    - 'CHEMBL:CHEMBL1833984' -> 'CHEMBL1833984'
    - 'CHEMBL1833984' -> 'CHEMBL1833984'
    - '5-FU' -> '5-FU'
    
    Returns normalized drug ID
    """
    drug_id = str(drug_id).strip()
    
    # Handle CHEMBL: prefix format
    if ':' in drug_id:
        parts = drug_id.split(':')
        # Return the part after the colon if it's a known database prefix
        if parts[0] in ['CHEMBL', 'DRUGBANK', 'PUBCHEM']:
            return parts[1].strip()
    
    return drug_id


def get_drug_targets(drug_name, drug_to_genes):
    """
    Get target genes for a specific drug
    """
    drug_name = str(drug_name).strip()
    drug_name_lower = drug_name.lower()
    
    if drug_name in drug_to_genes:
        return drug_to_genes[drug_name]
    
    # Try case-insensitive match
    for drug, genes in drug_to_genes.items():
        if drug.lower() == drug_name_lower:
            return genes
    
    # Try partial match
    for drug, genes in drug_to_genes.items():
        if drug_name_lower in drug.lower() or drug.lower() in drug_name_lower:
            return genes
    
    return []

In [22]:
drug_df['drug_name_normalized'] = drug_df['drug_name'].apply(normalize_drug_id)
drug_df[['drug_name','gene_name','interaction_score']]

,drug_name,gene_name,interaction_score
0,RACLOPRIDE,CYP2D6,0.017709
1,CHEMBL:CHEMBL1833984,PPARG,0.840123
2,CHEMBL:CHEMBL91609,ATAD5,0.177992
3,"3,4-DICHLOROISOCOUMARIN",RGS4,0.034319
4,WITHAFERIN A,MAPK1,0.050007
...,...,...,...
98234,MIRDAMETINIB,TP53,0.014489
98235,SULANEMADLIN,TP53,0.115911
98236,LY3009120,TP53,0.025758
98237,CABOZANTINIB S-MALATE,TP53,0.010537


In [26]:
drug_to_genes = build_drug_to_genes_mapping(drug_df)


Building drug-to-genes mapping...
Mapped 18854 drugs to genes


In [27]:
drug_targets = {}
for drug in all_drugs:
    targets = get_drug_targets(drug, drug_to_genes)
    if targets:
        drug_targets[drug] = targets


In [39]:
avg_length = sum(len(v) for v in drug_targets.values()) / len(drug_targets)
avg_length

14.58678323476472

In [30]:
3889/ 6245

0.622738190552442